In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import sys
sys.path.append('contrib/plots/')
from plot_utils import (
    create_figure, 
    style_axes, 
    apply_color_palette, 
    plot_line,
    set_publication_style,
    save_figure
)

In [ ]:
df = pd.read_csv("../../.local/data/sim_consolidated.csv")

In [ ]:
# plot 1: Absolute value vs. relative to baseline
llama_2_13b_df = df[df['model_id']=='meta-llama/Llama-2-13b-hf']
plot_1df = llama_2_13b_df[llama_2_13b_df['input_length']>1]
bsz_1_df = plot_1df[plot_1df['batch_size']==1]

sim_df = bsz_1_df[bsz_1_df['source']=='simulation']
real_df = bsz_1_df[bsz_1_df['source']=='real']

sim_baseline_time = sim_df[(sim_df['input_length'] == 64) & (sim_df['output_length'] == 16)]['prefill_time_s'].mean()
real_baseline_time = real_df[(real_df['input_length'] == 64) & (real_df['output_length'] == 16)]['prefill_time_s'].mean()

sim_df['relative_prefill_time_s'] = sim_df['prefill_time_s'] / sim_baseline_time
real_df['relative_prefill_time_s'] = real_df['prefill_time_s'] / real_baseline_time

# Convert to milliseconds
sim_df['prefill_time_ms'] = sim_df['prefill_time_s'] * 1000
real_df['prefill_time_ms'] = real_df['prefill_time_s'] * 1000

# Calculate statistics with confidence intervals for real data
def calculate_stats_with_ci(group, confidence=0.95):
    """Calculate mean and confidence interval for a group"""
    import scipy.stats as stats
    mean = group.mean()
    if len(group) > 1:
        sem = stats.sem(group)  # Standard error of the mean
        ci = stats.t.interval(confidence, len(group)-1, loc=mean, scale=sem)
        return mean, ci[0], ci[1]
    else:
        return mean, mean, mean

# Calculate means and confidence intervals for simulation and real data
sim_stats = sim_df.groupby('input_length')['prefill_time_ms'].apply(calculate_stats_with_ci).apply(pd.Series)
sim_stats.columns = ['mean', 'ci_lower', 'ci_upper']

real_stats = real_df.groupby('input_length')['prefill_time_ms'].apply(calculate_stats_with_ci).apply(pd.Series)
real_stats.columns = ['mean', 'ci_lower', 'ci_upper']

# For relative times
sim_rel_stats = sim_df.groupby('input_length')['relative_prefill_time_s'].apply(calculate_stats_with_ci).apply(pd.Series)
sim_rel_stats.columns = ['mean', 'ci_lower', 'ci_upper']

real_rel_stats = real_df.groupby('input_length')['relative_prefill_time_s'].apply(calculate_stats_with_ci).apply(pd.Series)
real_rel_stats.columns = ['mean', 'ci_lower', 'ci_upper']

set_publication_style(font_size=10)

# Create figure with two subplots side by side
fig, (ax1, ax2) = create_figure(format_type="double_column", ncols=2, nrows=1, figsize=(9, 3.75))

# Get colors from palette
colors = apply_color_palette("colorblind", 2)

# Plot 1: Absolute prefill times (in milliseconds)
plot_line(
    x=sim_stats.index.values,
    y=sim_stats['mean'].values,
    ax=ax1,
    color=colors[0],
    marker='o',
    linewidth=2,
    markersize=6,
    label='Simulation'
)

# Add connecting line for real data
ax1.plot(
    real_stats.index.values,
    real_stats['mean'].values,
    color=colors[1],
    linewidth=2,
    linestyle='-',
    label='Real'
)

# Add confidence intervals for real data
ax1.errorbar(
    x=real_stats.index.values,
    y=real_stats['mean'].values,
    yerr=[real_stats['mean'] - real_stats['ci_lower'], real_stats['ci_upper'] - real_stats['mean']],
    fmt='s',
    color=colors[1],
    linewidth=2,
    markersize=6,
    capsize=4,
    capthick=1.5,
    label=''  # No duplicate label
)

# Style the left subplot (absolute values)
style_axes(
    ax1,
    title="Absolute Prefill Time",
    xlabel="Input Length",
    ylabel="Time (ms)",
    grid=True,
    spine_style="default",
    legend=True
)

# Plot 2: Relative prefill times
plot_line(
    x=sim_rel_stats.index.values,
    y=sim_rel_stats['mean'].values,
    ax=ax2,
    color=colors[0],
    marker='o',
    linewidth=2,
    markersize=6,
    label='Simulation'
)

# Add connecting line for real data (relative)
ax2.plot(
    real_rel_stats.index.values,
    real_rel_stats['mean'].values,
    color=colors[1],
    linewidth=2,
    linestyle='-',
    label='Real'
)

# Add confidence intervals for real data (relative)
ax2.errorbar(
    x=real_rel_stats.index.values,
    y=real_rel_stats['mean'].values,
    yerr=[real_rel_stats['mean'] - real_rel_stats['ci_lower'], real_rel_stats['ci_upper'] - real_rel_stats['mean']],
    fmt='s',
    color=colors[1],
    linewidth=2,
    markersize=6,
    capsize=4,
    capthick=1.5,
    label=''  # No duplicate label
)

# Add horizontal line at y=1 for relative plot (baseline)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.7, linewidth=1)

# Style the right subplot (relative values)
style_axes(
    ax2,
    title="Relative Prefill Time",
    xlabel="Input Length",
    ylabel="Relative Time",
    grid=True,
    spine_style="default",
    legend=True
)

# Set x-axis to log scale if input lengths vary widely
all_input_lengths = list(sim_stats.index) + list(real_stats.index)
if max(all_input_lengths) / min(all_input_lengths) > 10:
    ax1.set_xscale('log')
    ax2.set_xscale('log')

plt.tight_layout()
plt.show()
save_figure(fig, "../../contrib/plots/figures/prefill_time_comparison", formats=["pdf"])

In [ ]:
# Double column figure: Absolute vs Relative Decode Times per Token by Batch Size
other_bsz_df = llama_2_13b_df[llama_2_13b_df['batch_size']>1]
sim_df = other_bsz_df[other_bsz_df['source']=='simulation']
real_df = other_bsz_df[other_bsz_df['source']=='real']

# Use batch_size=2 as baseline for relative comparison
sim_baseline_time = sim_df[(sim_df['batch_size'] == 2)]['decode_time_per_token_s'].mean()
real_baseline_time = real_df[(real_df['batch_size'] == 2)]['decode_time_per_token_s'].mean()

# Calculate relative values if not already present
if 'relative_decode_time_per_token_s' not in sim_df.columns:
    sim_df['relative_decode_time_per_token_s'] = sim_df['decode_time_per_token_s'] / sim_baseline_time
    real_df['relative_decode_time_per_token_s'] = real_df['decode_time_per_token_s'] / real_baseline_time

# add time_ms
sim_df['decode_time_per_token_ms'] = sim_df['decode_time_per_token_s'] * 1000
real_df['decode_time_per_token_ms'] = real_df['decode_time_per_token_s'] * 1000

# Calculate statistics with confidence intervals for real data
def calculate_stats_with_ci(group, confidence=0.95):
    """Calculate mean and confidence interval for a group"""
    import scipy.stats as stats
    mean = group.mean()
    if len(group) > 1:
        sem = stats.sem(group)  # Standard error of the mean
        ci = stats.t.interval(confidence, len(group)-1, loc=mean, scale=sem)
        return mean, ci[0], ci[1]
    else:
        return mean, mean, mean

# Calculate means and confidence intervals for simulation and real data
sim_stats = sim_df.groupby('batch_size')['decode_time_per_token_ms'].apply(calculate_stats_with_ci).apply(pd.Series)
sim_stats.columns = ['mean', 'ci_lower', 'ci_upper']

real_stats = real_df.groupby('batch_size')['decode_time_per_token_ms'].apply(calculate_stats_with_ci).apply(pd.Series)
real_stats.columns = ['mean', 'ci_lower', 'ci_upper']

# For relative times
sim_rel_stats = sim_df.groupby('batch_size')['relative_decode_time_per_token_s'].apply(calculate_stats_with_ci).apply(pd.Series)
sim_rel_stats.columns = ['mean', 'ci_lower', 'ci_upper']

real_rel_stats = real_df.groupby('batch_size')['relative_decode_time_per_token_s'].apply(calculate_stats_with_ci).apply(pd.Series)
real_rel_stats.columns = ['mean', 'ci_lower', 'ci_upper']

# Sort by batch_size for proper plotting
sim_stats = sim_stats.sort_index()
real_stats = real_stats.sort_index()
sim_rel_stats = sim_rel_stats.sort_index()
real_rel_stats = real_rel_stats.sort_index()

print("Real data statistics:")
print(real_stats)
print("\nSimulation data statistics:")
print(sim_stats)

set_publication_style(font_size=10)

# Create figure with two subplots side by side
fig, (ax1, ax2) = create_figure(format_type="double_column", ncols=2, nrows=1, figsize=(9, 3.75))

# Get colors from palette
colors = apply_color_palette("colorblind", 2)

# Plot 1: Absolute decode time per token (left figure)
plot_line(
    x=sim_stats.index.values,
    y=sim_stats['mean'].values,
    ax=ax1,
    color=colors[0],
    marker='o',
    linewidth=2,
    markersize=6,
    label='Simulation'
)

# Add connecting line for real data
ax1.plot(
    real_stats.index.values,
    real_stats['mean'].values,
    color=colors[1],
    linewidth=2,
    linestyle='-',
    label='Real'
)

# Add confidence intervals for real data
ax1.errorbar(
    x=real_stats.index.values,
    y=real_stats['mean'].values,
    yerr=[real_stats['mean'] - real_stats['ci_lower'], real_stats['ci_upper'] - real_stats['mean']],
    fmt='s',
    color=colors[1],
    linewidth=2,
    markersize=6,
    capsize=4,
    capthick=1.5,
    label=''  # No duplicate label
)

# Style the left subplot (absolute values)
style_axes(
    ax1,
    title="Absolute Decode Time",
    xlabel="Batch Size",
    ylabel="Time per Token (ms)",
    grid=True,
    spine_style="default",
    legend=True
)

# Plot 2: Relative decode time per token (right figure)
plot_line(
    x=sim_rel_stats.index.values,
    y=sim_rel_stats['mean'].values,
    ax=ax2,
    color=colors[0],
    marker='o',
    linewidth=2,
    markersize=6,
    label='Simulation'
)

# Add connecting line for real data (relative)
ax2.plot(
    real_rel_stats.index.values,
    real_rel_stats['mean'].values,
    color=colors[1],
    linewidth=2,
    linestyle='-',
    label='Real'
)

# Add confidence intervals for real data (relative)
ax2.errorbar(
    x=real_rel_stats.index.values,
    y=real_rel_stats['mean'].values,
    yerr=[real_rel_stats['mean'] - real_rel_stats['ci_lower'], real_rel_stats['ci_upper'] - real_rel_stats['mean']],
    fmt='s',
    color=colors[1],
    linewidth=2,
    markersize=6,
    capsize=4,
    capthick=1.5,
    label=''  # No duplicate label
)

# Add horizontal line at y=1 for relative plot (baseline)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.7, linewidth=1)

# Style the right subplot (relative values)
style_axes(
    ax2,
    title="Relative Decode Time",
    xlabel="Batch Size",
    ylabel="Rel. Time",
    grid=True,
    spine_style="default",
    legend=True
)

# Set x-axis to log scale if batch sizes vary widely
all_batch_sizes = list(sim_stats.index) + list(real_stats.index)
if max(all_batch_sizes) / min(all_batch_sizes) > 10:
    ax1.set_xscale('log')
    ax2.set_xscale('log')

plt.tight_layout()
plt.show()
save_figure(fig, "../../contrib/plots/figures/decode_time_by_batch_size_comparison", formats=["pdf"])

In [ ]:
def calculate_total_time(df_row):
    """Total time: prefill_time_s + decode_time_per_token_s * output_length"""
    return df_row['prefill_time_s'] + df_row['decode_time_per_token_s'] * df_row['output_length']

# Four setups for Llama-2-7b.  Currently 11/12 cells are populated;
# only RTX3090 Real @ IL=128/OL=64/BS=1 is missing and will render as a gap.
h100_real_7b    = df[(df['hw_name'] == 'NVDA:H100:NVL')      & (df['model_id'] == 'meta-llama/Llama-2-7b-hf') & (df['source'] == 'real')]
h100_sim_7b     = df[(df['hw_name'] == 'NVDA:H100:NVL')      & (df['model_id'] == 'meta-llama/Llama-2-7b-hf') & (df['source'] == 'simulation')]
rtx3090_real_7b = df[(df['hw_name'] == 'NVDA:RTX3090:PCIe')  & (df['model_id'] == 'meta-llama/Llama-2-7b-hf') & (df['source'] == 'real')        & (df['tp_size'] == 1)]
rtx3090_sim_7b  = df[(df['hw_name'] == 'NVDA:RTX3090:PCIe')  & (df['model_id'] == 'meta-llama/Llama-2-7b-hf') & (df['source'] == 'simulation') & (df['tp_size'] == 1)]

selected_configs = [
    (128,   64, 1),   # short prompt, short output, BS=1
    (512,  256, 4),   # medium prompt, medium output, BS=4
    (2048, 1024, 8),  # long prompt, long output, BS=8
]
selected_df = pd.DataFrame(selected_configs, columns=['input_length', 'output_length', 'batch_size'])
keys = ['input_length', 'output_length', 'batch_size']

# Compute mean total_time_s per config per setup; missing cells become NaN.
def mean_total_time(d):
    d = pd.merge(d, selected_df, on=keys, how='inner')
    if len(d) == 0:
        return pd.Series(dtype=float)
    d = d.copy()
    d['total_time_s'] = d.apply(calculate_total_time, axis=1)
    return d.groupby(keys)['total_time_s'].mean()

hetero_plot_df = pd.DataFrame({
    'h100_real':    mean_total_time(h100_real_7b),
    'h100_sim':     mean_total_time(h100_sim_7b),
    'rtx3090_real': mean_total_time(rtx3090_real_7b),
    'rtx3090_sim':  mean_total_time(rtx3090_sim_7b),
}).reset_index()

# Preserve selected_configs order.
hetero_plot_df = pd.merge(selected_df, hetero_plot_df, on=keys, how='left')
print(hetero_plot_df)
print("Missing cells (NaN):")
print(hetero_plot_df.set_index(keys).isna().sum().to_string())

In [ ]:
set_publication_style(font_size=10)
fig, (ax1, ax2) = create_figure(format_type="double_column", ncols=2, nrows=1, figsize=(9, 3.75))

colors = apply_color_palette("colorblind", 4)

hetero_plot_df['config'] = hetero_plot_df.apply(
    lambda row: f"IL{int(row['input_length'])}/OL{int(row['output_length'])}/BS{int(row['batch_size'])}",
    axis=1,
)
configs = hetero_plot_df['config'].tolist()
x = np.arange(len(configs))
width = 0.2

h100_real    = hetero_plot_df['h100_real'].values
h100_sim     = hetero_plot_df['h100_sim'].values
rtx3090_real = hetero_plot_df['rtx3090_real'].values
rtx3090_sim  = hetero_plot_df['rtx3090_sim'].values

# Plot 1: absolute E2E time
p1 = ax1.bar(x - 1.5*width, h100_real,    width, label="H100 Real",    color=colors[0], alpha=0.85, linewidth=1, edgecolor='k')
p2 = ax1.bar(x - 0.5*width, h100_sim,     width, label="H100 Sim",     color=colors[1], alpha=0.85, linewidth=1, edgecolor='k')
p3 = ax1.bar(x + 0.5*width, rtx3090_real, width, label="RTX3090 Real", color=colors[2], alpha=0.85, linewidth=1, edgecolor='k')
p4 = ax1.bar(x + 1.5*width, rtx3090_sim,  width, label="RTX3090 Sim",  color=colors[3], alpha=0.85, linewidth=1, edgecolor='k')

ax1.set_xlabel('Configuration')
ax1.set_ylabel('Time (s)')
ax1.set_title('E2E Time')
ax1.set_xticks(x)
ax1.set_xticklabels(configs, rotation=15)
ax1.grid(True, alpha=0.3, axis='y')

def add_value_labels(bars, ax, fmt='{:.2f}'):
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.annotate(fmt.format(h),
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=7)

for p in (p1, p2, p3, p4):
    add_value_labels(p, ax1, '{:.2f}')

# Plot 2: time normalized to H100 Real
norm_h100_real    = h100_real    / h100_real
norm_h100_sim     = h100_sim     / h100_real
norm_rtx3090_real = rtx3090_real / h100_real
norm_rtx3090_sim  = rtx3090_sim  / h100_real

q1 = ax2.bar(x - 1.5*width, norm_h100_real,    width, color=colors[0], alpha=0.85, linewidth=1, edgecolor='k')
q2 = ax2.bar(x - 0.5*width, norm_h100_sim,     width, color=colors[1], alpha=0.85, linewidth=1, edgecolor='k')
q3 = ax2.bar(x + 0.5*width, norm_rtx3090_real, width, color=colors[2], alpha=0.85, linewidth=1, edgecolor='k')
q4 = ax2.bar(x + 1.5*width, norm_rtx3090_sim,  width, color=colors[3], alpha=0.85, linewidth=1, edgecolor='k')

ax2.axhline(y=1, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax2.set_xlabel('Configuration')
ax2.set_ylabel('Time / H100 Real')
ax2.set_title('Relative E2E Time')
ax2.set_xticks(x)
ax2.set_xticklabels(configs, rotation=15)
ax2.grid(True, alpha=0.3, axis='y')

for p in (q1, q2, q3, q4):
    add_value_labels(p, ax2, '{:.2f}')

fig.legend(
    handles=[p1[0], p2[0], p3[0], p4[0]],
    labels=["H100 Real", "H100 Sim", "RTX3090 Real", "RTX3090 Sim"],
    ncols=4,
    bbox_to_anchor=(0.5, 1.04),
    loc='center',
)
sns.despine()
plt.tight_layout()
plt.show()
save_figure(fig, "../../contrib/plots/figures/h100_rtx3090_four_setup_comparison", formats=["pdf"])